# Colab flow

### GPU usage

To use GPU in colab: `Edit > Notebook settings or Runtime>Change runtime type and select GPU as Hardware accelerator`.

### Setting repo

In [ ]:
colab = False

For `colab=True` you need to rewrite the `branch` and `github_token` below.

In [ ]:
if colab:
    !pip install condacolab # Handling conda environment installation
    import condacolab
    condacolab.install()
    !git clone -b branch https://github_token@github.com/Nixtla/nixtlats.git
    %cd nixtlats
    !mamba env update -n base -f environment.yml
    !pip install -e .

# Standard flow

In [ ]:
import numpy as np
import pytorch_lightning as pl
import torch as t
from pytorch_lightning import loggers as pl_loggers
from pytorch_lightning import seed_everything

from nixtla.data.datasets.m4 import M4, M4Info
from nixtla.data.tsdataset import TimeSeriesDataset
from nixtla.data.tsloader import TimeSeriesLoader
from nixtla.models.esrnn.esrnn import ESRNN

In [ ]:
seed_everything(117982, workers=True)

## Import Quarterly data

In [ ]:
group = M4Info['Quarterly']
Y_df, _, S_df = M4.load(directory='data', group=group.name)

In [ ]:
train_ts_dataset = TimeSeriesDataset(Y_df=Y_df, S_df=S_df,
                                     ds_in_test=group.horizon,
                                     mode='full',
                                     window_sampling_limit=20_000, # To limit backprop time
                                     input_size=8,
                                     output_size=group.horizon,
                                     idx_to_sample_freq=1,
                                     len_sample_chunks=group.horizon * 4,
                                     complete_inputs=True, 
                                     skip_nonsamplable=True)

In [ ]:
test_ts_dataset = TimeSeriesDataset(Y_df=Y_df, S_df=S_df,
                                    ds_in_test=0,
                                    mode='full',
                                    window_sampling_limit=500_000, # To limit backprop time
                                    input_size=8,
                                    output_size=group.horizon,
                                    idx_to_sample_freq=1,
                                    complete_inputs=True,
                                    complete_outputs=True,
                                    len_sample_chunks=group.horizon * 3,
                                    skip_nonsamplable=False,
                                    last_samplable_window=True)

In [ ]:
train_ts_loader = TimeSeriesLoader(dataset=train_ts_dataset,
                                   batch_size=512,
                                   eq_batch_size=True,
                                   num_workers=8,
                                   shuffle=False)

In [ ]:
test_ts_loader = TimeSeriesLoader(dataset=test_ts_dataset,
                                  batch_size=1024,
                                  eq_batch_size=False,
                                  num_workers=8,
                                  shuffle=False)

In [ ]:
model = ESRNN(n_series=group.n_ts,
          n_x=0, n_s=1,
          idx_to_sample_freq=1,
          input_size=8,
          output_size=group.horizon,
          learning_rate=1e-3,
          lr_scheduler_step_size=10,
          lr_decay=0.1,
          per_series_lr_multip=1,
          gradient_clipping_threshold=20,
          rnn_weight_decay=0,
          level_variability_penalty=80,
          testing_percentile=50,
          training_percentile=45,
          cell_type='LSTM',
          state_hsize=40,
          dilations=[[1, 2], [4, 8]],
          add_nl_layer=False,
          loss='SMYL',
          val_loss='SMAPE',
          seasonality=[4])

In [ ]:
trainer = pl.Trainer(max_epochs=15, gpus=int(colab),
                     progress_bar_refresh_rate=5, deterministic=True)
trainer.fit(model, train_ts_loader)

In [ ]:
outputs = trainer.predict(model, test_ts_loader)

In [ ]:
_, y_hat = zip(*outputs)

In [ ]:
y_hat = t.cat(y_hat)[:, -1].cpu().numpy()

In [ ]:
y = Y_df.groupby('unique_id').tail(group.horizon)['y'].values.reshape(-1, group.horizon)

In [ ]:
from nixtla.losses.numpy import smape

In [ ]:
smape(y, y_hat)